# 09B · Build a system card as a release artifact

**Objective (10 min):** a model card alone is too narrow for a RAG/tool agent. Use Hugging Face's
`ModelCard` utility to create valid card metadata and Markdown, but document the **complete system**:
model, prompt, retrieval, tools, policy, privacy, evaluation, monitoring, and human oversight — and
pull the numbers from the evidence the earlier labs produced.

In [ ]:
# --- Workshop bootstrap: run this cell first ------------------------------------
# JupyterLab starts every kernel inside the notebook's own folder. Move to the
# toolkit root so shared modules (demo_agent, workshop_utils) import and the
# _evidence/ output paths resolve, no matter where Jupyter was launched from.
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "workshop_utils.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Workshop root:", ROOT)

In [ ]:
import json
from pathlib import Path
from textwrap import dedent

from huggingface_hub import ModelCard, ModelCardData


def load_json(path: str, default: dict) -> dict:
    p = Path(path)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else default

baseline = load_json("_evidence/00_baseline_and_contract.json", {"metrics": {"constrained_attack_success_rate": "not run"}, "release_gate": {"passed": False}})
trace_evidence = load_json("_evidence/08_redacted_trace.json", {"incident_packet": {"detected": "not run", "affected_implementations": []}})
fairness = load_json("_evidence/09_fairness_assessment.json", {"equalized_odds_difference": "not run"})
garak = load_json("_evidence/02_garak_summary.json", {"summary": []})

In [ ]:
card_data = ModelCardData(
    language=["en"],
    license="apache-2.0",
    library_name="responsible-ai-workshop",
    tags=["rag", "agent", "security-evaluation", "system-card"],
)

asr = baseline["metrics"].get("constrained_attack_success_rate", "not run")
release_passed = baseline["release_gate"].get("passed", False)
incident = trace_evidence["incident_packet"]
garak_line = ", ".join(f"{r['target']}: {r['attack_success_rate']:.0%}" for r in garak["summary"]) or "not run"
eod = fairness.get("equalized_odds_difference")
eod_str = f"{eod:.3f}" if isinstance(eod, float) else str(eod)

content = dedent(f"""
---
{card_data.to_yaml()}
card_type: ai-system-card
status: workshop-example
---

# System Card: Constrained Support RAG Agent v1.1

## Decision summary

**Workshop release decision:** {"PASS for the included synthetic corpus" if release_passed else "BLOCK"}.
**Owner:** Agent platform lead.
**Expiry:** Re-evaluate after any model, prompt, retriever, knowledge source, tool, policy, or data-use change.

## Intended use and affected users

The system answers synthetic customer-support return questions and drafts refund decisions. It is intended to assist a human-supported workflow. Users and customers are affected by answer quality, privacy, refusal behavior, and refund handling.

## Out-of-scope and prohibited use

Not approved for autonomous payments, legal/medical/credit decisions, unrestricted web or shell access, real customer data, cross-tenant retrieval, or use without the application-side policy and approval controls.

## System architecture and boundaries

- Deterministic workshop agent; no external model call in the default path.
- Mixed-trust retrieval corpus with source trust labels.
- Pydantic-typed action proposals; Guardrails AI validators on free-text fields.
- Server-side policy: high-value refund requires human approval bound to the proposal hash.
- No model-generated command reaches an executor directly; one-time capabilities with replay protection.
- Redacted, decision-oriented OpenTelemetry traces; raw prompts are not span attributes.

## Data and privacy

All bundled examples use synthetic data. Names, e-mail, Indian mobile numbers, internal customer IDs, and canaries are transformed with Presidio before model/log destinations. Evidence contains hashes and redacted views, not raw direct identifiers. Production use would require purpose, processor, residency, retention, deletion, access, and data-subject workflow decisions.

## Security controls

- Threat model as code (`pytm`) with explicit trust boundaries; LLM01/02/05/07/09 addressed, LLM03/08 residual.
- Direct secret-request denial; retrieved instructions treated as untrusted data.
- Typed action allow-list, tenant checks, amount threshold, approval binding, one-time capability, replay protection.
- Deterministic prompt-injection and excessive-agency regression corpus (`pytest` + Inspect AI).
- Model artifacts quarantined and scanned with ModelScan before loading.
- Kill switches: disable tool, force approval, revoke capability/credential, disable source, read-only mode.

## Evaluation evidence

- Constrained attack-success rate on bundled corpus: **{asr}**.
- garak `promptinject.HijackHateHumans` attack-success rate — {garak_line}.
- Hard gates: zero canary leaks, zero unauthorized side effects, benign return answer includes the documented 30-day policy.
- Telemetry incident detected in the bundled run: **{incident.get("detected")}** (implementations: {incident.get("affected_implementations")}).
- Evidence is synthetic and small; it does not establish general robustness.

## Fairness and accessibility

A separate synthetic assessment shows aggregate accuracy hiding a higher false-negative rate for one interaction language (equalized-odds difference **{eod_str}**). Production assessment must identify domain-specific harms, relevant groups/intersections, sample sizes, uncertainty, accessibility needs, and mitigation trade-offs.

## Human oversight and appeals

High-value refunds require an identified human approver bound to the exact proposal. A production UI must show source, amount, reason, uncertainty, and proposed action; support rejection, correction, and customer escalation; and never dark-pattern approval.

## Monitoring and incident response

Monitor canary leakage, foreign-tenant source IDs, unexpected tools, policy bypass, approval mismatch, replay, false-refusal/utility drift, subgroup error drift, and trace/export failures. Preserve redacted traces and restricted evidence references. Convert confirmed incidents into stable evaluation IDs before re-enable.

## Known limitations and residual risks

- Rule-based workshop target is not representative of open-ended model behavior.
- Included attacks are not exhaustive; multilingual/encoded/multi-turn coverage is limited.
- Pattern PII recognition can miss context-specific or malformed identifiers and over-redact useful text.
- Source trust labels, tenant metadata, identity, and approvals must themselves be protected from tampering.
- A clean static artifact scan or red-team run cannot prove absence of vulnerabilities.

## Change management

Re-run threat modeling, privacy review, security/utility evals, subgroup assessment, supply-chain scan, and incident tabletop for material changes. Deploy by immutable version/digest and retain a tested rollback path.

## Evidence owners and approval

- Product outcome owner: **TBD in production**
- Engineering/control owner: **TBD in production**
- Security review: **TBD in production**
- Privacy/legal/compliance review where applicable: **TBD in production**
""").strip() + "\n"

card = ModelCard(content)
print(card.data.to_dict())

In [ ]:
required_sections = [
    "## Intended use and affected users",
    "## Out-of-scope and prohibited use",
    "## System architecture and boundaries",
    "## Data and privacy",
    "## Security controls",
    "## Evaluation evidence",
    "## Fairness and accessibility",
    "## Human oversight and appeals",
    "## Monitoring and incident response",
    "## Known limitations and residual risks",
    "## Change management",
]
missing = [heading for heading in required_sections if heading not in card.content]
assert not missing, missing
assert "TBD in production" in card.content, "Do not invent approvals or owners."
assert "not run" not in card.content, "Run modules 00, 02B, 08 and 09A first so the card cites real evidence."

In [ ]:
out = Path("_evidence/09_SYSTEM_CARD.md")
out.parent.mkdir(exist_ok=True)
card.save(out)
print("PASS: system card contains required evidence and explicit limitations")
print("Wrote", out.resolve())
print("\nPreview:\n")
print("\n".join(out.read_text(encoding="utf-8").splitlines()[:30]))

## Review discipline

A card should be generated from versioned evidence where possible, reviewed by accountable people, and
rejected when fields are unknown — not filled with confident fiction. Publish the audience-appropriate
card while keeping sensitive attack details and raw evidence access-controlled. See
`SYSTEM_CARD_TEMPLATE.md` for the blank template.